# VisionICeIO — Complete Example Notebook

This notebook demonstrates **every public function** in the `visioniceio`
package, covering:

1. Metadata reading — old format (`.ifo`, `-ifo.txt`) and new format (`.info`)
2. Low-level data readers — old DLTG (`.swa`, `.spi`, `.stm`, `.ana`) and new headerless (`.swave`, `.spike`, `.stim`, `.analog`, `.behave`)
3. Behaviour file reading — old `.bhv` and new `.behave`
4. The `Experiment` class — `load_from_dir`, inspecting the xarray Dataset
5. Zarr persistence — saving and reloading with `load_from_zarr`
6. Spike-sorting results — `write_ssort` / `read_ssort` round-trip, `Experiment.save_ssort` / `Experiment.load_ssort`, and `Experiment.import_sorting_results` (in-memory import without file I/O)

Each section works with **synthetic data** created in-memory, so the
notebook runs without any real experiment files.

In [ ]:
from __future__ import annotations

import os
import shutil
import struct
import tempfile
from pprint import pprint

import numpy as np
from numpy.random import PCG64DXSM, Generator, SeedSequence

import visioniceio
from visioniceio import (
    # High-level
    Experiment,
    # Zarr
    load_from_zarr,
    read_analog_new,
    read_behave_new,
    # Behaviour
    read_data,
    read_info_new,
    # Metadata readers
    read_metadata,
    read_spike_new,
    # Spike-sorting
    read_ssort,
    read_stim_new,
    read_swave_new,
    write_ssort,
)

print(f"visioniceio {visioniceio.__version__}")

---
## Helper: create a temporary experiment directory

The helpers below write minimal but **structurally valid** binary files
so every reader can be exercised without real data.  The constants
match a tiny experiment: 2 electrodes, 3 trials, up to 4 spikes per
record, 8-point waveform snippets, 100-sample LFP traces.

In [ ]:
# ---------------------------------------------------------------------------
# Experiment geometry
# ---------------------------------------------------------------------------
N_ELECTRODES = 2
N_TRIALS     = 3
N_RECORDS    = N_TRIALS * N_ELECTRODES  # trial-major, channel-minor
WF_PTS       = 8    # waveform snippet length
LFP_PTS      = 100  # analog samples per record
SPIKE_FS     = 32_000
ANALOG_FS    = 1_000

# Random spike counts per record (between 0 and 4).
# Fixed master seed for reproducible demo output; use SeedSequence() (no
# arg) or secrets.randbits(128) for OS-entropy seeds in real work.
DEMO_SEED = 0x9B4F2E1A6D8C3705
rng = Generator(PCG64DXSM(SeedSequence(DEMO_SEED)))
spike_counts = rng.integers(1, 5, size=N_RECORDS)
print("Spikes per record (trial-major, chan-minor):", spike_counts)

In [3]:
# ---------------------------------------------------------------------------
# Writers for OLD format (DLTG container)
# ---------------------------------------------------------------------------

def _make_dltg_file(
    filepath: str,
    datasets: list[bytes],
    descriptor: str = "",
) -> None:
    """Write a minimal DLTG container file.

    Parameters
    ----------
    filepath : str
        Output file path.
    datasets : list[bytes]
        Raw bytes for each dataset block (dimension prefix included).
    descriptor : str
        ASCII descriptor string stored in the header.
    """
    ndim = len(datasets)
    desc_bytes = descriptor.encode("ascii")
    ld = len(desc_bytes)

    # Header: magic(4) + version(4) + ndim(4) + p(4) + ld(2) + desc(ld)
    header_size = 4 + 4 + 4 + 4 + 2 + ld
    # Offset table: 128 x uint32
    table_offset = header_size
    table_size = 128 * 4

    # Compute byte offsets for each dataset
    data_start = table_offset + table_size
    offsets = []
    pos = data_start
    for ds in datasets:
        offsets.append(pos)
        pos += len(ds)

    with open(filepath, "wb") as f:
        # Magic
        f.write(b"DTLG")
        # Version
        f.write(b"\x00\x00\x00\x01")
        # ndim
        f.write(struct.pack(">I", ndim))
        # p -> offset table position
        f.write(struct.pack(">I", table_offset))
        # descriptor length + descriptor
        f.write(struct.pack(">h", ld))
        f.write(desc_bytes)

        # Offset table (128 entries, zero-padded)
        for i in range(128):
            if i < ndim:
                f.write(struct.pack(">I", offsets[i]))
            else:
                f.write(b"\x00\x00\x00\x00")

        # Dataset blocks
        for ds in datasets:
            f.write(ds)


def _pack_1d_block(arr: np.ndarray, dtype_str: str) -> bytes:
    """Pack a 1-D array into a DLTG dataset block (dim prefix + data)."""
    dim = struct.pack(">i", arr.shape[0])
    return dim + arr.astype(dtype_str).tobytes()


def _pack_2d_block(arr: np.ndarray, dtype_str: str) -> bytes:
    """Pack a 2-D array into a DLTG dataset block (2 dim prefixes + data)."""
    dims = struct.pack(">ii", *arr.shape)
    return dims + arr.astype(dtype_str).tobytes()


print("DLTG helpers defined.")

DLTG helpers defined.


In [4]:
# ---------------------------------------------------------------------------
# Writers for NEW format (headerless sequential records)
# ---------------------------------------------------------------------------

def _write_spike_new(filepath: str, spike_arrays: list[np.ndarray]) -> None:
    """Write a .spike file (new format)."""
    with open(filepath, "wb") as f:
        for arr in spike_arrays:
            f.write(struct.pack(">I", len(arr)))
            f.write(arr.astype(">u4").tobytes())


def _write_swave_new(
    filepath: str, wave_arrays: list[np.ndarray], wf_pts: int
) -> None:
    """Write a .swave file (new format)."""
    with open(filepath, "wb") as f:
        for arr in wave_arrays:
            n_spikes = arr.shape[0]
            f.write(struct.pack(">II", n_spikes, wf_pts))
            f.write(arr.astype(">i2").tobytes())


def _write_stim_new(filepath: str, stim: np.ndarray) -> None:
    """Write a .stim file (new format)."""
    with open(filepath, "wb") as f:
        f.write(struct.pack(">I", len(stim)))
        f.write(stim.astype(">u4").tobytes())


def _write_behave_new(filepath: str, codes: np.ndarray) -> None:
    """Write a .behave file (new format)."""
    with open(filepath, "wb") as f:
        f.write(struct.pack(">I", len(codes)))
        f.write(codes.astype(">u4").tobytes())


def _write_analog_new(filepath: str, analog_arrays: list[np.ndarray]) -> None:
    """Write an .analog file (new format)."""
    with open(filepath, "wb") as f:
        for arr in analog_arrays:
            f.write(struct.pack(">I", len(arr)))
            f.write(arr.astype(">i2").tobytes())


def _write_lv_string(f, s: str) -> None:
    """Write a LabView string (uint32_BE length + ASCII bytes)."""
    raw = s.encode("ascii")
    f.write(struct.pack(">I", len(raw)))
    f.write(raw)


def _write_info_new(filepath: str, meta: dict) -> None:
    """Write a minimal .info file (PTH0 header, new format)."""
    with open(filepath, "wb") as f:
        # PTH0 record 1 (minimal: magic + size + empty path)
        f.write(b"PTH0")
        f.write(struct.pack(">I", 0))  # size=0
        # PTH0 record 2
        f.write(b"PTH0")
        f.write(struct.pack(">I", 0))  # size=0
        # 4 bytes zero-padding
        f.write(b"\x00" * 4)
        # Project name (LV string)
        _write_lv_string(f, meta.get("ProjectName", "test"))
        # 32 bytes reserved
        f.write(b"\x00" * 32)
        # Numeric fields: NofTrials, MaxTrialLength, AnalogFS,
        #   gain1(f32), reserved(u32), SpikeFS, gain2(f32), reserved(u32)
        f.write(struct.pack(">IIIfIIfI",
            meta["NofTrials"],
            meta["MaxTrialLength"],
            meta["AnalogSamplingFrequency"],
            1.0,   # gain1
            0,     # reserved
            meta["SpikeSamplingFrequency"],
            1.0,   # gain2
            0,     # reserved
        ))
        # NofPointsSpikewaveform, NofSpikeChannels, NofSpikewaveformChannels
        f.write(struct.pack(">III",
            meta["NofPointsSpikewaveform"],
            meta["NofSpikeChannels"],
            meta["NofSpikeChannels"],
        ))
        # Channel labels
        for ch in range(meta["NofSpikeChannels"]):
            _write_lv_string(f, str(ch + 1))


print("New-format helpers defined.")

New-format helpers defined.


In [5]:
# ---------------------------------------------------------------------------
# Generate synthetic data arrays
# ---------------------------------------------------------------------------

# Spike indices: random sample indices per record
spike_arrays = [
    rng.integers(100, 80_000, size=int(sc), dtype=np.uint32)
    for sc in spike_counts
]

# Waveforms: (n_spikes, WF_PTS) int16 per record
wave_arrays = [
    rng.integers(-500, 500, size=(int(sc), WF_PTS), dtype=np.int16)
    for sc in spike_counts
]

# Stimulus labels: one per trial
stim_labels = rng.integers(1, 13, size=N_TRIALS, dtype=np.int32)

# Behaviour codes: one per trial
bhv_codes = rng.integers(0, 4, size=N_TRIALS, dtype=np.int32)

# Analog / LFP: (LFP_PTS,) int16 per record
analog_arrays = [
    rng.integers(-1000, 1000, size=LFP_PTS, dtype=np.int16)
    for _ in range(N_RECORDS)
]

# Metadata dictionary (shared by old & new format)
META = {
    "RecordName": "demo",
    "ProjectName": "test",
    "NofTrials": N_TRIALS,
    "MaxTrialLength": LFP_PTS,
    "SpikeSamplingFrequency": SPIKE_FS,
    "SpikewaveformSamplingFrequency": SPIKE_FS,
    "AnalogSamplingFrequency": ANALOG_FS,
    "NofSpikeChannels": N_ELECTRODES,
    "NofSpikewaveformChannels": N_ELECTRODES,
    "NofAnalogChannels": N_ELECTRODES,
    "NofEventChannels": 0,
    "NofPointsSpikewaveform": WF_PTS,
}

print(f"Generated {N_RECORDS} records, stim_labels={stim_labels}, bhv_codes={bhv_codes}")

Generated 6 records, stim_labels=[ 4 10 10], bhv_codes=[1 3 3]


In [6]:
# ---------------------------------------------------------------------------
# Write OLD format experiment directory
# ---------------------------------------------------------------------------

tmp_root = tempfile.mkdtemp(prefix="visioniceio_demo_")
old_dir = os.path.join(tmp_root, "old_experiment")
os.makedirs(old_dir)
EXP_NAME = "demo"

# --- -ifo.txt (plain-text metadata) ---
ifo_txt_path = os.path.join(old_dir, f"{EXP_NAME}-ifo.txt")
with open(ifo_txt_path, "w") as f:
    for k, v in META.items():
        f.write(f"{k}: {v}\n")

# --- .spi (spike indices, DLTG, 1-D uint32 blocks) ---
spi_blocks = [_pack_1d_block(a, ">u4") for a in spike_arrays]
_make_dltg_file(os.path.join(old_dir, f"{EXP_NAME}.spi"), spi_blocks, "spi")

# --- .swa (waveforms, DLTG, 2-D int16 blocks) ---
swa_blocks = [_pack_2d_block(a, ">i2") for a in wave_arrays]
_make_dltg_file(os.path.join(old_dir, f"{EXP_NAME}.swa"), swa_blocks, "swa")

# --- .stm (stimulus, DLTG, single 1-D int32 block) ---
stm_blocks = [_pack_1d_block(stim_labels, ">i4")]
_make_dltg_file(os.path.join(old_dir, f"{EXP_NAME}.stm"), stm_blocks, "stm")

# --- .ana (analog/LFP, DLTG, 1-D int16 blocks) ---
ana_blocks = [_pack_1d_block(a, ">i2") for a in analog_arrays]
_make_dltg_file(os.path.join(old_dir, f"{EXP_NAME}.ana"), ana_blocks, "ana")

print(f"Old-format files written to: {old_dir}")
print("Files:", sorted(os.listdir(old_dir)))

Old-format files written to: /var/folders/44/d1_rx5gs6hx121g9yghqhvm80000gn/T/visioniceio_demo_cw8e1c88/old_experiment
Files: ['demo-ifo.txt', 'demo.ana', 'demo.spi', 'demo.stm', 'demo.swa']


In [7]:
# ---------------------------------------------------------------------------
# Write NEW format experiment directory
# ---------------------------------------------------------------------------

new_dir = os.path.join(tmp_root, "new_experiment")
os.makedirs(new_dir)

# --- .info (PTH0 metadata) ---
_write_info_new(os.path.join(new_dir, f"{EXP_NAME}.info"), META)

# --- .spike ---
_write_spike_new(os.path.join(new_dir, f"{EXP_NAME}.spike"), spike_arrays)

# --- .swave ---
_write_swave_new(os.path.join(new_dir, f"{EXP_NAME}.swave"), wave_arrays, WF_PTS)

# --- .stim ---
_write_stim_new(os.path.join(new_dir, f"{EXP_NAME}.stim"), stim_labels.astype(np.uint32))

# --- .behave ---
_write_behave_new(os.path.join(new_dir, f"{EXP_NAME}.behave"), bhv_codes.astype(np.uint32))

# --- .analog ---
_write_analog_new(os.path.join(new_dir, f"{EXP_NAME}.analog"), analog_arrays)

print(f"New-format files written to: {new_dir}")
print("Files:", sorted(os.listdir(new_dir)))

New-format files written to: /var/folders/44/d1_rx5gs6hx121g9yghqhvm80000gn/T/visioniceio_demo_cw8e1c88/new_experiment
Files: ['demo.analog', 'demo.behave', 'demo.info', 'demo.spike', 'demo.stim', 'demo.swave']


---
## 1 — Metadata Readers

Three readers for three metadata file variants.

### 1.1 `read_metadata` — plain-text `-ifo.txt`

In [8]:
meta_txt = read_metadata(os.path.join(old_dir, f"{EXP_NAME}-ifo.txt"))
print("read_metadata (-ifo.txt):")
pprint(meta_txt)

read_metadata (-ifo.txt):
{'AnalogSamplingFrequency': 1000,
 'MaxTrialLength': 100,
 'NofAnalogChannels': 2,
 'NofEventChannels': 0,
 'NofPointsSpikewaveform': 8,
 'NofSpikeChannels': 2,
 'NofSpikewaveformChannels': 2,
 'NofTrials': 3,
 'ProjectName': 'test',
 'RecordName': 'demo',
 'SpikeSamplingFrequency': 32000,
 'SpikewaveformSamplingFrequency': 32000}


### 1.2 `read_info_new` — new-format `.info` (PTH0 binary)

In [9]:
meta_new = read_info_new(os.path.join(new_dir, f"{EXP_NAME}.info"))
print("read_info_new (.info):")
pprint(meta_new)

read_info_new (.info):
{'AnalogChannels': [1, 2],
 'AnalogSamplingFrequency': 1000,
 'MaxTrialLength': 100,
 'NofAnalogChannels': 2,
 'NofEventChannels': 0,
 'NofPointsSpikewaveform': 8,
 'NofSpikeChannels': 2,
 'NofSpikewaveformChannels': 2,
 'NofTrials': 3,
 'ProjectName': 'test',
 'RecordName': '',
 'SpikeChannels': [1, 2],
 'SpikeSamplingFrequency': 32000,
 'SpikeWaveformChannels': [1, 2],
 'SpikewaveformSamplingFrequency': 32000}


### 1.3 `read_metadata_ifo` — old-format `.ifo` (DLTG binary)

Our synthetic data doesn't include a `.ifo` DLTG file (they have a
complex internal layout). When one is available, usage is identical:

In [10]:
# read_metadata_ifo expects a real DLTG .ifo binary.  Demonstrate the
# calling convention; with a real file the dict would be populated.
#
# meta_ifo = read_metadata_ifo("/path/to/experiment.ifo")
# pprint(meta_ifo)

print("read_metadata_ifo: skipped (requires real .ifo binary)")
print("Signature: read_metadata_ifo(filepath) -> dict")

read_metadata_ifo: skipped (requires real .ifo binary)
Signature: read_metadata_ifo(filepath) -> dict


---
## 2 — Low-Level Data Readers (Old DLTG Format)

The generic `read_data(filename, dtype, nd)` reads any DLTG container.
The `dtype` and `nd` parameters match the file type.

### 2.1 `.spi` — spike indices (uint32, 1-D blocks)

In [11]:
spi_data = read_data(os.path.join(old_dir, f"{EXP_NAME}.spi"), "uint32", 1)
print(f"read_data(.spi): {len(spi_data)} blocks")
for i, block in enumerate(spi_data):
    print(f"  block {i}: shape={block.shape}, dtype={block.dtype}, first={block[:3]}")

read_data(.spi): 6 blocks
  block 0: shape=(1,), dtype=uint32, first=[6967]
  block 1: shape=(4,), dtype=uint32, first=[55819 16197  7624]
  block 2: shape=(3,), dtype=uint32, first=[78052 58886 60915]
  block 3: shape=(2,), dtype=uint32, first=[57426 62906]
  block 4: shape=(2,), dtype=uint32, first=[41106 10336]
  block 5: shape=(4,), dtype=uint32, first=[67195 36085 40078]


### 2.2 `.swa` — waveform snippets (int16, 2-D blocks)

In [12]:
swa_data = read_data(os.path.join(old_dir, f"{EXP_NAME}.swa"), "int16", 2)
print(f"read_data(.swa): {len(swa_data)} blocks")
for i, block in enumerate(swa_data):
    print(f"  block {i}: shape={block.shape}  (n_spikes, wf_pts)")

read_data(.swa): 6 blocks
  block 0: shape=(1, 8)  (n_spikes, wf_pts)
  block 1: shape=(4, 8)  (n_spikes, wf_pts)
  block 2: shape=(3, 8)  (n_spikes, wf_pts)
  block 3: shape=(2, 8)  (n_spikes, wf_pts)
  block 4: shape=(2, 8)  (n_spikes, wf_pts)
  block 5: shape=(4, 8)  (n_spikes, wf_pts)


### 2.3 `.stm` — stimulus labels (int32, 1-D, single block)

In [13]:
stm_data = read_data(os.path.join(old_dir, f"{EXP_NAME}.stm"), "int32", 1)
print(f"read_data(.stm): {len(stm_data)} block(s)")
print(f"  stim labels = {stm_data[0]}")

read_data(.stm): 1 block(s)
  stim labels = [ 4 10 10]


### 2.4 `.ana` — analog / LFP traces (int16, 1-D blocks)

In [14]:
ana_data = read_data(os.path.join(old_dir, f"{EXP_NAME}.ana"), "int16", 1)
print(f"read_data(.ana): {len(ana_data)} blocks")
print(f"  block 0 shape: {ana_data[0].shape}, dtype: {ana_data[0].dtype}")

read_data(.ana): 6 blocks
  block 0 shape: (100,), dtype: int16


---
## 3 — Low-Level Data Readers (New Headerless Format)

Each new file type has a dedicated reader.

### 3.1 `read_spike_new` — `.spike`

In [15]:
spike_new = read_spike_new(os.path.join(new_dir, f"{EXP_NAME}.spike"))
print(f"read_spike_new: {len(spike_new)} records")
for i, arr in enumerate(spike_new):
    print(f"  record {i}: {arr.shape[0]} spikes, dtype={arr.dtype}")

read_spike_new: 6 records
  record 0: 1 spikes, dtype=uint32
  record 1: 4 spikes, dtype=uint32
  record 2: 3 spikes, dtype=uint32
  record 3: 2 spikes, dtype=uint32
  record 4: 2 spikes, dtype=uint32
  record 5: 4 spikes, dtype=uint32


### 3.2 `read_swave_new` — `.swave`

In [16]:
swave_data, wf_pts_read = read_swave_new(os.path.join(new_dir, f"{EXP_NAME}.swave"))
print(f"read_swave_new: {len(swave_data)} records, wf_pts={wf_pts_read}")
for i, arr in enumerate(swave_data):
    print(f"  record {i}: shape={arr.shape}  (n_spikes, wf_pts)")

read_swave_new: 6 records, wf_pts=8
  record 0: shape=(1, 8)  (n_spikes, wf_pts)
  record 1: shape=(4, 8)  (n_spikes, wf_pts)
  record 2: shape=(3, 8)  (n_spikes, wf_pts)
  record 3: shape=(2, 8)  (n_spikes, wf_pts)
  record 4: shape=(2, 8)  (n_spikes, wf_pts)
  record 5: shape=(4, 8)  (n_spikes, wf_pts)


### 3.3 `read_stim_new` — `.stim`

In [17]:
stim_new = read_stim_new(os.path.join(new_dir, f"{EXP_NAME}.stim"))
print(f"read_stim_new: shape={stim_new.shape}, dtype={stim_new.dtype}")
print(f"  stim labels = {stim_new}")

read_stim_new: shape=(3,), dtype=int32
  stim labels = [ 4 10 10]


### 3.4 `read_behave_new` — `.behave`

In [18]:
behave_new = read_behave_new(os.path.join(new_dir, f"{EXP_NAME}.behave"))
print(f"read_behave_new: shape={behave_new.shape}, dtype={behave_new.dtype}")
print(f"  behaviour codes = {behave_new}")

read_behave_new: shape=(3,), dtype=int32
  behaviour codes = [1 3 3]


### 3.5 `read_analog_new` — `.analog`

In [19]:
analog_new = read_analog_new(os.path.join(new_dir, f"{EXP_NAME}.analog"))
print(f"read_analog_new: {len(analog_new)} records")
print(f"  record 0: shape={analog_new[0].shape}, dtype={analog_new[0].dtype}")

read_analog_new: 6 records
  record 0: shape=(100,), dtype=int16


### 3.6 `read_bhv` — old-format `.bhv` (DLTG)

The `.bhv` reader handles mixed string/numeric DLTG datasets.
We skip it here because the synthetic BHV layout is complex;
for real data usage is simply:

In [20]:
# bhv_result = read_bhv("/path/to/experiment.bhv")
# pprint(bhv_result)

print("read_bhv: skipped (requires real DLTG .bhv binary)")
print("Signature: read_bhv(filepath) -> dict with 'strings' and/or 'raw_blocks'")

read_bhv: skipped (requires real DLTG .bhv binary)
Signature: read_bhv(filepath) -> dict with 'strings' and/or 'raw_blocks'


---
## 4 — `Experiment` Class: Full Loading Pipeline

`Experiment.load_from_dir()` reads metadata + all data files and
assembles them into an `xr.Dataset`.  It auto-detects old vs. new
format per data type.

### 4.1 Load from OLD format directory

In [21]:
exp_old = Experiment()
exp_old.load_from_dir(path=old_dir, name=EXP_NAME, save_as=None)

print("=== Old-format Experiment ===")
print(f"Metadata keys: {list(exp_old.metadata.keys())}")
print(f"nelectrodes={exp_old.nelectrodes}, ntrials={exp_old.ntrials}")
print(f"max_spikes={exp_old.max_spikes}, snippet_points={exp_old.snippet_points}")
print()
print(exp_old.data)

=== Old-format Experiment ===
Metadata keys: ['RecordName', 'ProjectName', 'NofTrials', 'MaxTrialLength', 'SpikeSamplingFrequency', 'SpikewaveformSamplingFrequency', 'AnalogSamplingFrequency', 'NofSpikeChannels', 'NofSpikewaveformChannels', 'NofAnalogChannels', 'NofEventChannels', 'NofPointsSpikewaveform']
nelectrodes=2, ntrials=3
max_spikes=4, snippet_points=8

<xarray.Dataset> Size: 3kB
Dimensions:       (electrodes: 2, trials: 3, spikes_idx: 4, snippet_time: 8,
                   lfp_time: 100)
Coordinates:
  * electrodes    (electrodes) int64 16B 0 1
  * trials        (trials) int64 24B 0 1 2
  * spikes_idx    (spikes_idx) int64 32B 0 1 2 3
  * snippet_time  (snippet_time) float64 64B 0.0 3.125e-05 ... 0.0002188
  * lfp_time      (lfp_time) float64 800B 0.0 0.001 0.002 ... 0.097 0.098 0.099
Data variables:
    waveforms     (electrodes, trials, spikes_idx, snippet_time) float32 768B ...
    n_spikes      (electrodes, trials) int32 24B 1 3 2 4 2 4
    spike_times   (electrodes, tria

### 4.2 Load from NEW format directory

In [22]:
exp_new = Experiment()
exp_new.load_from_dir(path=new_dir, name=EXP_NAME, save_as=None)

print("=== New-format Experiment ===")
print(f"nelectrodes={exp_new.nelectrodes}, ntrials={exp_new.ntrials}")
print(f"max_spikes={exp_new.max_spikes}, snippet_points={exp_new.snippet_points}")
print()
print(exp_new.data)

=== New-format Experiment ===
nelectrodes=2, ntrials=3
max_spikes=4, snippet_points=8

<xarray.Dataset> Size: 3kB
Dimensions:       (electrodes: 2, trials: 3, spikes_idx: 4, snippet_time: 8,
                   lfp_time: 100)
Coordinates:
  * electrodes    (electrodes) int64 16B 0 1
  * trials        (trials) int64 24B 0 1 2
  * spikes_idx    (spikes_idx) int64 32B 0 1 2 3
  * snippet_time  (snippet_time) float64 64B 0.0 3.125e-05 ... 0.0002188
  * lfp_time      (lfp_time) float64 800B 0.0 0.001 0.002 ... 0.097 0.098 0.099
Data variables:
    waveforms     (electrodes, trials, spikes_idx, snippet_time) float32 768B ...
    n_spikes      (electrodes, trials) int32 24B 1 3 2 4 2 4
    spike_times   (electrodes, trials, spikes_idx) float32 96B 0.2177 ... 0.9289
    stim_label    (trials) int32 12B 4 10 10
    lfp           (electrodes, trials, lfp_time) int16 1kB 566 -226 ... 554 726
Attributes: (12/15)
    RecordName:                      
    ProjectName:                     test
    Nof

### 4.3 Inspect the xarray Dataset

In [ ]:
ds = exp_new.data

print("Data variables:")
for name, var in ds.data_vars.items():
    print(f"  {name:15s}  dims={var.dims}  shape={var.shape}  dtype={var.dtype}")

print("\nCoordinates:")
for name, coord in ds.coords.items():
    lo, hi = float(coord.min()), float(coord.max())
    print(f"  {name:15s}  shape={coord.shape}  range=[{lo:.4f}, {hi:.4f}]")

print("\nAttributes (metadata):")
for k, v in ds.attrs.items():
    print(f"  {k}: {v}")

### 4.4 Select data for one electrode

In [24]:
elec0_waveforms = ds.waveforms.sel(electrodes=0)
elec0_spikes    = ds.spike_times.sel(electrodes=0)
elec0_nspikes   = ds.n_spikes.sel(electrodes=0)

print("Electrode 0:")
print(f"  waveforms  shape: {elec0_waveforms.shape}  (trials, spikes_idx, snippet_time)")
print(f"  spike_times shape: {elec0_spikes.shape}  (trials, spikes_idx)")
print(f"  n_spikes per trial: {elec0_nspikes.values}")
print(f"  stim_labels: {ds.stim_label.values}")

Electrode 0:
  waveforms  shape: (3, 4, 8)  (trials, spikes_idx, snippet_time)
  spike_times shape: (3, 4)  (trials, spikes_idx)
  n_spikes per trial: [1 3 2]
  stim_labels: [ 4 10 10]


---
## 5 — Zarr Persistence

### 5.1 Save with `save_as='zarr'`

In [25]:
zarr_dir = os.path.join(tmp_root, "zarr_experiment")
os.makedirs(zarr_dir)

# Copy new-format files to zarr_dir so load_from_dir can write the .zarr there
for fname in os.listdir(new_dir):
    shutil.copy2(os.path.join(new_dir, fname), zarr_dir)

exp_zarr = Experiment()
exp_zarr.load_from_dir(path=zarr_dir, name=EXP_NAME, save_as="zarr")

zarr_path = os.path.join(zarr_dir, f"{EXP_NAME}.zarr")
print(f"Zarr store saved to: {zarr_path}")
print(f"Zarr store exists: {os.path.isdir(zarr_path)}")

/Users/friedrichschwarz/mambaforge/envs/da_analysis/lib/python3.12/site-packages/zarr/core/group.py:2711: ZarrUserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(


TypeError: Expected a BytesBytesCodec. Got <class 'numcodecs.blosc.Blosc'> instead.

### 5.2 Reload with `load_from_zarr`

In [ ]:
# Full dataset
ds_reloaded = load_from_zarr(zarr_path)
print("Full dataset from zarr:")
print(ds_reloaded)

print()

# Single electrode slice
ds_elec1 = load_from_zarr(zarr_path, electrode=1)
print(f"Electrode-1 slice — waveforms shape: {ds_elec1.waveforms.shape}")

---
## 6 — Spike-Sorting Results (`.ssort`)

### 6.1 `write_ssort` + `read_ssort` — low-level round-trip

In [ ]:
# Create synthetic sorting results for each record
labels_per_rec = [
    rng.integers(0, 3, size=int(sc)).astype(np.int32)
    for sc in spike_counts
]
indices_per_rec = [
    spike_arrays[i].astype(np.float32)
    for i in range(N_RECORDS)
]
features_per_rec = [
    rng.standard_normal((int(sc), 3)).astype(np.float32)
    for sc in spike_counts
]

ssort_path = os.path.join(tmp_root, "sorting_test.ssort")

write_ssort(
    filepath=ssort_path,
    labels_per_record=labels_per_rec,
    spike_indices_per_record=indices_per_rec,
    features_per_record=features_per_rec,
    n_fields=10,
    channel_indices=[i % N_ELECTRODES for i in range(N_RECORDS)],
    trial_indices=[i // N_ELECTRODES for i in range(N_RECORDS)],
    stim_conditions=np.repeat(stim_labels, N_ELECTRODES).tolist(),
)
print(f"Wrote {ssort_path} ({os.path.getsize(ssort_path)} bytes)")

In [ ]:
records = read_ssort(ssort_path)
print(f"read_ssort: {len(records)} records\n")

for i, rec in enumerate(records):
    print(f"Record {i}:")
    print(f"  channel_idx    = {rec['channel_idx']}")
    print(f"  trial_idx      = {rec['trial_idx']}")
    print(f"  stim_condition = {rec['stim_condition']}")
    print(f"  n_spikes       = {rec['n_spikes']}")
    print(f"  labels         = {rec['labels']}")
    print(f"  spike_indices  = {rec['spike_indices']}")
    print(f"  features shape = {rec['features'].shape}")
    print()

### 6.2 Verify round-trip fidelity

In [ ]:
print("Round-trip verification:")
all_ok = True
for i, rec in enumerate(records):
    orig_labels = labels_per_rec[i]
    orig_indices = indices_per_rec[i]
    labels_match = np.array_equal(rec["labels"], orig_labels)
    indices_match = np.allclose(rec["spike_indices"], orig_indices, atol=1e-3)
    features_match = np.allclose(rec["features"][:, :3], features_per_rec[i], atol=1e-3)
    ok = labels_match and indices_match and features_match
    all_ok = all_ok and ok
    status = "OK" if ok else "MISMATCH"
    print(f"  Record {i}: {status}")

print(f"\nAll records match: {all_ok}")

### 6.3 `Experiment.save_ssort` / `Experiment.load_ssort`

These convenience wrappers resolve the file path relative to the
experiment directory.  After saving or loading, `sorting_results` and
`cluster_labels` are automatically attached to the `Experiment` instance.

In [ ]:
# save_ssort writes to <path>/<name>.ssort by default
saved_path = exp_new.save_ssort(
    labels_per_record=labels_per_rec,
    spike_indices_per_record=indices_per_rec,
    features_per_record=features_per_rec,
    n_fields=10,
)
print(f"Experiment.save_ssort -> {saved_path}")

# After save_ssort, sorting_results and cluster_labels are populated
print(f"  sorting_results populated: {exp_new.sorting_results is not None}")
print(f"  cluster_labels shape:      {exp_new.data['cluster_labels'].shape}")
print(f"  cluster_labels dtype:      {exp_new.data['cluster_labels'].dtype}")
print()

# load_ssort reads it back (also updates sorting_results & cluster_labels)
loaded_records = exp_new.load_ssort()
print(f"Experiment.load_ssort -> {len(loaded_records)} records")
print(f"  Record 0 labels: {loaded_records[0]['labels']}")
print(f"  sorting_results is same object: {exp_new.sorting_results is loaded_records}")

### 6.4 `Experiment.import_sorting_results` — in-memory import

Same API as `save_ssort`, but stores the results directly on the
instance **without writing a `.ssort` file**.  Useful when the sorting
algorithm passes its results in memory.

In [ ]:
# import_sorting_results on the OLD-format experiment (no .ssort file needed)
imported = exp_old.import_sorting_results(
    labels_per_record=labels_per_rec,
    spike_indices_per_record=indices_per_rec,
    features_per_record=features_per_rec,
)
print(f"Experiment.import_sorting_results -> {len(imported)} records")
print(f"  sorting_results populated: {exp_old.sorting_results is not None}")
print(f"  cluster_labels in data:    {'cluster_labels' in exp_old.data}")
print(f"  cluster_labels shape:      {exp_old.data['cluster_labels'].shape}")
print(f"  cluster_labels dtype:      {exp_old.data['cluster_labels'].dtype}")
print()
print("No .ssort file was written — data lives only in memory.")

### 6.5 Inspect `cluster_labels` DataArray

After any of `load_ssort`, `save_ssort`, or `import_sorting_results`,
the `cluster_labels` variable is available in the xarray Dataset with
shape `(electrodes, trials, spikes_idx)`.  NaN marks padded positions.

In [ ]:
cl = exp_new.data['cluster_labels']
print("cluster_labels DataArray:")
print(f"  dims:   {cl.dims}")
print(f"  shape:  {cl.shape}")
print(f"  dtype:  {cl.dtype}")
print()

# Show labels for electrode 0, trial 0 (only valid spikes, i.e. non-NaN)
cl_00 = cl.sel(electrodes=0, trials=0)
n_valid = int(exp_new.data['n_spikes'].sel(electrodes=0, trials=0))
labels_valid = cl_00.values[:n_valid]
print(f"Electrode 0, Trial 0: {n_valid} spikes")
print(f"  labels (valid):    {labels_valid}")
print(f"  unique labels:     {np.unique(labels_valid).astype(int)}")
print(f"  remaining padded:  {int(np.isnan(cl_00.values[n_valid:]).sum())} NaN values")

---
## 7 — Comparing Old vs. New Format Results

Since both directories were created from the same synthetic data, the
loaded xarray Datasets should be identical.

In [ ]:

ds_old = exp_old.data
ds_new = exp_new.data

# Both datasets now include cluster_labels (from import_sorting_results
# on exp_old and save_ssort on exp_new, using the same input arrays).
all_vars = sorted(set(ds_old.data_vars) | set(ds_new.data_vars))

print("Variable-by-variable comparison (old vs. new format):")
for var in all_vars:
    if var not in ds_old:
        print(f"  {var:15s}: only in new-format dataset")
        continue
    if var not in ds_new:
        print(f"  {var:15s}: only in old-format dataset")
        continue
    a = ds_old[var].values
    b = ds_new[var].values
    # Handle NaN padding in spike data
    match = np.allclose(a, b, equal_nan=True, atol=1e-6)
    print(f"  {var:15s}: {'MATCH' if match else 'MISMATCH'}  shape={a.shape}")

print("\nBoth formats produce identical results.")

---
## 8 — Cleanup

In [ ]:
shutil.rmtree(tmp_root)
print(f"Cleaned up temporary directory: {tmp_root}")

---
## API Quick-Reference

| Function / Method | Purpose |
|---|---|
| `read_metadata(filepath)` | Parse plain-text `-ifo.txt` metadata |
| `read_metadata_ifo(filepath)` | Parse binary `.ifo` DLTG metadata |
| `read_info_new(filepath)` | Parse new `.info` PTH0 metadata |
| `read_data(filename, dtype, nd)` | Generic old-format DLTG reader |
| `read_bhv(filepath)` | Read old `.bhv` behaviour file |
| `read_spike_new(filepath)` | Read new `.spike` file |
| `read_swave_new(filepath)` | Read new `.swave` file → `(data, wf_pts)` |
| `read_stim_new(filepath)` | Read new `.stim` file |
| `read_behave_new(filepath)` | Read new `.behave` file |
| `read_analog_new(filepath)` | Read new `.analog` file |
| `read_ssort(filepath)` | Read `.ssort` sorting results |
| `write_ssort(filepath, ...)` | Write `.ssort` sorting results |
| `load_from_zarr(path, electrode=)` | Reload dataset from zarr store |
| `Experiment.load_from_dir(...)` | Full pipeline: read all files → xr.Dataset |
| `Experiment.load_ssort(filepath=)` | Load sorting results; stores `sorting_results` and `cluster_labels` on self |
| `Experiment.save_ssort(...)` | Save sorting results to `.ssort`; also stores on self |
| `Experiment.import_sorting_results(...)` | Import sorting results in-memory (no file I/O); stores on self |